# Detailed Model Testing on Google Colab

This notebook runs comprehensive testing with detailed statistics on multiple datasets (AVDeepfake1M, ShareVeo3, Sora2).

Features:
- Test on multiple datasets separately
- Comprehensive metrics (Accuracy, Precision, Recall, F1, AUROC)
- Confusion matrices
- Per-batch statistics
- Probability analysis
- JSON output for all results


In [ ]:
# Mount Google Drive to access your data files
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Configure paths - UPDATE THESE TO MATCH YOUR GOOGLE DRIVE STRUCTURE
HDF5_FILE_PATH = "/content/drive/MyDrive/MIT/Lab/deepfake_embeddings_2.h5"
CHECKPOINT_PATH = "/content/drive/MyDrive/MIT/Lab/best_model.pt"
MODEL_FILE_PATH = "/content/drive/MyDrive/MIT/Lab/time_series_model.py"
TEST_SCRIPT_PATH = "/content/drive/MyDrive/MIT/Lab/test_model_detailed.py"
OUTPUT_DIR = "/content/drive/MyDrive/MIT/Lab/test_results"

print("="*60)
print("CONFIGURED PATHS")
print("="*60)
print(f"HDF5 file: {HDF5_FILE_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Model file: {MODEL_FILE_PATH}")
print(f"Test script: {TEST_SCRIPT_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print("="*60)


In [ ]:
# Copy files to Colab workspace
import shutil
import os

# Create directories
os.makedirs("/content/models", exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Copy time_series_model.py
if os.path.exists(MODEL_FILE_PATH):
    shutil.copy(MODEL_FILE_PATH, "/content/models/time_series_model.py")
    print("✓ Copied time_series_model.py to /content/models/")
else:
    print(f"⚠️  Not found: {MODEL_FILE_PATH}")
    print("Please upload time_series_model.py to Google Drive and update MODEL_FILE_PATH")

# Copy test script
if os.path.exists(TEST_SCRIPT_PATH):
    shutil.copy(TEST_SCRIPT_PATH, "/content/test_model_detailed.py")
    print("✓ Copied test_model_detailed.py to /content/")
else:
    print(f"⚠️  Not found: {TEST_SCRIPT_PATH}")
    print("Please upload test_model_detailed.py to Google Drive and update TEST_SCRIPT_PATH")

print(f"\n✓ Output directory created: {OUTPUT_DIR}")


In [ ]:
# Install required packages
!pip install h5py numpy scikit-learn tqdm

# Install PyTorch with CUDA support (for GPU)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


In [ ]:
# Verify GPU is available
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  WARNING: No GPU detected! Testing will be slower.")
    print("Make sure you've selected a GPU runtime: Runtime > Change runtime type > GPU")


In [ ]:
# Verify files exist
import os

print("Checking required files...")
files_ok = True

if os.path.exists(HDF5_FILE_PATH):
    file_size = os.path.getsize(HDF5_FILE_PATH) / 1e9
    print(f"✓ HDF5 file found: {HDF5_FILE_PATH} ({file_size:.2f} GB)")
else:
    print(f"⚠️  HDF5 file NOT found: {HDF5_FILE_PATH}")
    files_ok = False

if os.path.exists(CHECKPOINT_PATH):
    file_size = os.path.getsize(CHECKPOINT_PATH) / 1e6
    print(f"✓ Checkpoint found: {CHECKPOINT_PATH} ({file_size:.2f} MB)")
    
    # Quick checkpoint info
    try:
        checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
        print(f"  Epoch: {checkpoint.get('epoch', 'N/A')}")
        if 'val_auroc' in checkpoint:
            print(f"  Validation AUROC: {checkpoint['val_auroc']:.4f}")
    except Exception as e:
        print(f"  ⚠️  Error reading checkpoint: {e}")
else:
    print(f"⚠️  Checkpoint NOT found: {CHECKPOINT_PATH}")
    files_ok = False

if os.path.exists("/content/models/time_series_model.py"):
    print("✓ Model file copied successfully")
else:
    print("⚠️  Model file not copied")
    files_ok = False

if os.path.exists("/content/test_model_detailed.py"):
    print("✓ Test script copied successfully")
else:
    print("⚠️  Test script not copied")
    files_ok = False

if files_ok:
    print("\n✅ All files ready!")
else:
    print("\n⚠️  Please check missing files above")


## Run Tests

Choose one of the options below to test on different datasets.


In [ ]:
# OPTION 1: Test on AVDeepfake1M and ShareVeo3 (default)
# This is the recommended option to see how your model performs on both training datasets

import sys
sys.path.insert(0, '/content/models')  # Add models directory to path

!python /content/test_model_detailed.py \
    --hdf5_path "{HDF5_FILE_PATH}" \
    --checkpoint_path "{CHECKPOINT_PATH}" \
    --output_dir "{OUTPUT_DIR}" \
    --batch_size 16


In [ ]:
# OPTION 2: Test only on AVDeepfake1M
# Uncomment the code below to run

# import sys
# sys.path.insert(0, '/content/models')
# 
# !python /content/test_model_detailed.py \
#     --hdf5_path "{HDF5_FILE_PATH}" \
#     --checkpoint_path "{CHECKPOINT_PATH}" \
#     --datasets avdeepfake1m \
#     --output_dir "{OUTPUT_DIR}" \
#     --batch_size 16


In [ ]:
# OPTION 3: Test only on ShareVeo3
# Uncomment the code below to run

# import sys
# sys.path.insert(0, '/content/models')
# 
# !python /content/test_model_detailed.py \
#     --hdf5_path "{HDF5_FILE_PATH}" \
#     --checkpoint_path "{CHECKPOINT_PATH}" \
#     --datasets shareveo3 \
#     --output_dir "{OUTPUT_DIR}" \
#     --batch_size 16


In [ ]:
# OPTION 4: Test on all three datasets (AVDeepfake1M, ShareVeo3, Sora2)
# Uncomment the code below to run

# import sys
# sys.path.insert(0, '/content/models')
# 
# !python /content/test_model_detailed.py \
#     --hdf5_path "{HDF5_FILE_PATH}" \
#     --checkpoint_path "{CHECKPOINT_PATH}" \
#     --datasets avdeepfake1m shareveo3 sora2 \
#     --output_dir "{OUTPUT_DIR}" \
#     --batch_size 16


In [ ]:
# OPTION 5: Test with per-sample predictions saved
# This saves predictions for the first 1000 samples for detailed analysis
# Uncomment the code below to run

# import sys
# sys.path.insert(0, '/content/models')
# 
# !python /content/test_model_detailed.py \
#     --hdf5_path "{HDF5_FILE_PATH}" \
#     --checkpoint_path "{CHECKPOINT_PATH}" \
#     --output_dir "{OUTPUT_DIR}" \
#     --batch_size 16 \
#     --save_predictions


## View Results


In [ ]:
# List all result files
import json
import os
from datetime import datetime

print("="*60)
print("RESULT FILES")
print("="*60)

if os.path.exists(OUTPUT_DIR):
    result_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.json')]
    if result_files:
        print(f"\nFound {len(result_files)} result file(s):\n")
        for f in sorted(result_files):
            file_path = os.path.join(OUTPUT_DIR, f)
            file_size = os.path.getsize(file_path) / 1024  # KB
            print(f"  • {f} ({file_size:.2f} KB)")
    else:
        print("\nNo result files found yet. Run the test cells above first.")
else:
    print(f"\nOutput directory not found: {OUTPUT_DIR}")


In [ ]:
# Load and display results from a specific file
# Update the filename to match your result file

import json
import os

# Get the most recent result file
if os.path.exists(OUTPUT_DIR):
    result_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.json')]
    if result_files:
        # Sort by modification time and get the latest
        result_files_with_time = [(f, os.path.getmtime(os.path.join(OUTPUT_DIR, f))) 
                                   for f in result_files]
        result_files_with_time.sort(key=lambda x: x[1], reverse=True)
        latest_file = result_files_with_time[0][0]
        
        file_path = os.path.join(OUTPUT_DIR, latest_file)
        
        print(f"Loading: {latest_file}\n")
        print("="*60)
        
        with open(file_path, 'r') as f:
            result = json.load(f)
        
        # Display summary
        print(f"Dataset: {result.get('dataset_name', 'Unknown')}")
        print(f"Number of samples: {result.get('num_samples', 0):,}")
        print(f"Number of segments: {result.get('num_segments', 0):,}")
        
        print(f"\nMetrics:")
        metrics = result.get('metrics', {})
        print(f"  Loss:      {metrics.get('loss', 0):.4f}")
        print(f"  Accuracy:  {metrics.get('accuracy', 0):.4f} ({metrics.get('accuracy', 0)*100:.2f}%)")
        print(f"  Precision: {metrics.get('precision', 0):.4f}")
        print(f"  Recall:    {metrics.get('recall', 0):.4f}")
        print(f"  F1 Score:  {metrics.get('f1_score', 0):.4f}")
        print(f"  AUROC:     {metrics.get('auroc', 0):.4f}")
        
        # Confusion matrix
        cm = result.get('confusion_matrix', {})
        print(f"\nConfusion Matrix:")
        print(f"  True Negatives (TN):  {cm.get('true_negative', 0):,}")
        print(f"  False Positives (FP): {cm.get('false_positive', 0):,}")
        print(f"  False Negatives (FN): {cm.get('false_negative', 0):,}")
        print(f"  True Positives (TP):  {cm.get('true_positive', 0):,}")
        
        # Label distribution
        print(f"\nLabel Distribution:")
        for label, count in result.get('label_distribution', {}).items():
            label_name = "Real" if int(label) == 1 else "Fake"
            pct = 100 * count / result.get('num_segments', 1)
            print(f"  {label_name}: {count:,} ({pct:.2f}%)")
        
        print("="*60)
    else:
        print("No result files found. Run the test cells above first.")
else:
    print(f"Output directory not found: {OUTPUT_DIR}")


In [ ]:
# Compare results across datasets (if you tested multiple)
# This cell loads all result files and creates a comparison table

import json
import os
import pandas as pd

if os.path.exists(OUTPUT_DIR):
    result_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.json') and 'combined' not in f]
    
    if len(result_files) > 1:
        print("="*70)
        print("COMPARISON ACROSS DATASETS")
        print("="*70)
        
        all_results = []
        for f in result_files:
            file_path = os.path.join(OUTPUT_DIR, f)
            with open(file_path, 'r') as file:
                result = json.load(file)
                dataset_name = result.get('dataset_name', 'unknown')
                metrics = result.get('metrics', {})
                all_results.append({
                    'Dataset': dataset_name,
                    'Samples': result.get('num_samples', 0),
                    'Segments': result.get('num_segments', 0),
                    'Loss': metrics.get('loss', 0),
                    'Accuracy': metrics.get('accuracy', 0),
                    'Precision': metrics.get('precision', 0),
                    'Recall': metrics.get('recall', 0),
                    'F1': metrics.get('f1_score', 0),
                    'AUROC': metrics.get('auroc', 0),
                })
        
        df = pd.DataFrame(all_results)
        print("\n" + df.to_string(index=False))
        print("\n" + "="*70)
    else:
        print("Need at least 2 result files to compare. Test multiple datasets first.")
else:
    print(f"Output directory not found: {OUTPUT_DIR}")


In [ ]:
# Download result files to your local machine
from google.colab import files
import os

if os.path.exists(OUTPUT_DIR):
    result_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.json')]
    
    if result_files:
        print(f"Found {len(result_files)} result file(s) to download:\n")
        for f in result_files:
            file_path = os.path.join(OUTPUT_DIR, f)
            print(f"  • {f}")
        
        print("\n⬇️  Downloading files...")
        for f in result_files:
            file_path = os.path.join(OUTPUT_DIR, f)
            files.download(file_path)
            print(f"  ✓ Downloaded {f}")
        
        print("\n✅ All files downloaded!")
    else:
        print("No result files found. Run the test cells above first.")
else:
    print(f"Output directory not found: {OUTPUT_DIR}")


## Notes

1. **GPU Runtime**: Make sure you've selected a GPU runtime (Runtime > Change runtime type > GPU) for faster testing

2. **File Paths**: Update the paths in Cell 2 to match your Google Drive structure:
   - If your files are in a different location, update `HDF5_FILE_PATH`, `CHECKPOINT_PATH`, etc.

3. **Batch Size**: If you get out-of-memory errors, reduce `--batch_size` from 16 to 8 or 4

4. **Results Location**: All results are saved to Google Drive in `OUTPUT_DIR`, so they persist even after the Colab session ends

5. **Testing Options**: 
   - **Option 1 (Cell 8)**: Tests on AVDeepfake1M + ShareVeo3 (recommended)
   - **Option 2 (Cell 9)**: Tests only on AVDeepfake1M
   - **Option 3 (Cell 10)**: Tests only on ShareVeo3
   - **Option 4 (Cell 11)**: Tests on all three datasets
   - **Option 5 (Cell 12)**: Includes per-sample predictions

## Setup Checklist

- [ ] Mounted Google Drive (Cell 1)
- [ ] Updated paths in Cell 2 to match your Drive structure
- [ ] Uploaded required files to Google Drive:
  - [ ] `deepfake_embeddings_2.h5`
  - [ ] `best_model.pt`
  - [ ] `time_series_model.py`
  - [ ] `test_model_detailed.py`
- [ ] Selected GPU runtime (Runtime > Change runtime type > GPU)
- [ ] Run cells in order up to Cell 6 to verify setup
- [ ] Run one of the test options (Cells 8-12)
